# road-damage-shift — Semaine 2 : Q1, combien d'images cibles annotées pour s'adapter à un pays jamais vu ?

Point de départ : le modèle entraîné en S1 sur Japon + Inde + Tchéquie (`best.pt`). On le fine-tune sur
0 / 50 / 200 / 500 images annotées des États-Unis (3 tirages chacune) et sur toute la réserve (plafond),
puis on mesure, avec IC bootstrap : la performance sur le **test États-Unis figé**, et l'**oubli** sur le test source.

**Protocole fixé avant les résultats** (section `fewshot` de `configs/default.yaml`) : 30 epochs pour tous,
lr = lr S1 / 10, pas de validation cible (on évalue `last.pt`), seed d'entraînement fixe.
Le coût d'annotation est **estimé** avec un modèle publié (7,8 s/image + 7 à 34,5 s/boîte), pas mesuré.

**Réglages Kaggle**
1. *Accelerator* : GPU T4 x1 ; *Internet* : On.
2. *Add Input* : ton dataset privé contenant les poids S1 (`best.pt` de YOLO11s, et de RT-DETR-l si tu veux aussi le traiter).
   Le notebook les trouve tout seul sous `/kaggle/input` (voir la cellule Paramètres).
3. `SMOKE = True` d'abord (≈ 10 min) ; puis `SMOKE = False` via *Save Version → Save & Run All*.

In [ ]:
# ---- Paramètres -------------------------------------------------------------
REPO_URL = "https://github.com/<TON_USER>/road-damage-shift.git"  # à remplacer
REPO_REF = "main"
REPO_DIR = "/tmp/road-damage-shift"
RAW = "/tmp/rdd_raw"
DATA_OUT = "/tmp/rdd_yolo"
SMOKE = True                 # True : k=0 et k=50 (1 tirage), 1 epoch ; False : protocole complet
S1_MANIFEST_SHA256 = "aff1da5af0e1c61885e4a59ddc058d8f659638117f99ab5d1f5df525cc57e2da"  # données de S1
# Poids S1 : chemin explicite, ou None pour chercher "<modèle>*best*.pt" / ".../<modèle>/best.pt" sous /kaggle/input
WEIGHTS = {"yolo11s": None, "rtdetr_l": None}

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf {REPO_DIR} && git clone -q --depth 1 -b {REPO_REF} {REPO_URL} {REPO_DIR}
!pip install -q -e "{REPO_DIR}[train]" faster-coco-eval

In [ ]:
import csv, hashlib, json, os, platform, subprocess, sys, time
from pathlib import Path

sys.path.insert(0, f"{REPO_DIR}/src")
import mlflow, pandas as pd, torch, ultralytics, yaml
from ultralytics import RTDETR, YOLO, settings

from roadshift.eval.report import report
from roadshift.fewshot.plan import (aggregate_curve, annotation_cost_hours, count_annotations, finetune_lr0,
                                    plan_runs, write_fewshot_yaml)
from roadshift.fewshot.plot import plot_curve

CFG = yaml.safe_load(open(f"{REPO_DIR}/configs/default.yaml"))
FS, TRAIN, CLASSES = CFG["fewshot"], CFG["train"], CFG["data"]["classes"]
GIT_SHA = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
VERSIONS = {"torch": torch.__version__, "ultralytics": ultralytics.__version__, "mlflow": mlflow.__version__,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}
print(GIT_SHA[:10], VERSIONS)
assert torch.cuda.is_available(), "Active le GPU T4 dans les réglages du notebook"

## 0–1. Données : mêmes téléchargement, conversion et découpages qu'en S1 (vérifiés par empreinte)

In [ ]:
!python -m roadshift.data.download --out {RAW} --countries Japan India Czech United_States
!python -m roadshift.data.convert --config {REPO_DIR}/configs/default.yaml --raw-root {RAW} --out {DATA_OUT} --overwrite
DATA = Path(DATA_OUT)
DATASET_INFO = json.load(open(DATA / "dataset_info.json"))
assert DATASET_INFO["manifest_sha256"] == S1_MANIFEST_SHA256, "Découpages différents de S1 : comparaison invalide"
print("Découpages identiques à S1 :", S1_MANIFEST_SHA256[:12])

## 2. Poids S1 et coût d'annotation estimé de chaque sous-ensemble
Le coût est calculé sur les **vrais** nombres d'images et de boîtes de chaque liste ; seuls les temps
unitaires viennent de la littérature (Su et al. 2012 ; Papadopoulos et al., ICCV 2017).

In [ ]:
def find_weights(key):
    if WEIGHTS.get(key):
        return Path(WEIGHTS[key])
    cands = [p for p in Path("/kaggle/input").rglob("*.pt") if key in str(p) and "best" in str(p)]
    return cands[0] if len(cands) == 1 else None

def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

MODELS = [m for m in FS["models"] if find_weights(m)]
missing = [m for m in FS["models"] if m not in MODELS]
assert MODELS, f"Aucun poids S1 trouvé pour {FS['models']} : ajoute le dataset des poids en Input"
if missing:
    print("Ignorés (poids absents) :", missing)
S1W = {m: find_weights(m) for m in MODELS}
for m, p in S1W.items():
    print(m, p, sha256(p)[:12])

pool_size = len((DATA / f"list_pool_{FS['target']}.txt").read_text().split())
RUNS_PLAN = plan_runs(FS, pool_size)
if SMOKE:
    MODELS = MODELS[:1]
    RUNS_PLAN = [r for r in RUNS_PLAN if r.k == 0 or (r.k == min(FS["sizes"]) and r.seed == FS["seeds"][0])]

cost_rows = []
for r in RUNS_PLAN:
    if r.k == 0:
        cost_rows.append({"run": r.name, "k": 0, "seed": None, "n_images": 0, "n_boxes": 0, "h_low": 0.0, "h_high": 0.0})
        continue
    c = count_annotations(DATA / r.list_name, DATA / "labels", CLASSES)
    lo, hi = annotation_cost_hours(c["n_images"], c["n_boxes"], FS["annotation_cost"])
    cost_rows.append({"run": r.name, "k": r.k, "seed": r.seed, **c, "h_low": round(lo, 2), "h_high": round(hi, 2)})
COST = pd.DataFrame(cost_rows)
display(COST)

## 3. MLflow

In [ ]:
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for k in ("MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if (v := _secret(k)):
        os.environ[k] = v
os.chdir("/kaggle/working")
mlflow.set_tracking_uri(CFG["mlflow"]["tracking_uri"] or _secret("MLFLOW_TRACKING_URI") or "sqlite:////kaggle/working/mlflow.db")
EXPERIMENT = CFG["mlflow"]["experiment"] + "-fewshot" + ("-smoke" if SMOKE else "")
mlflow.set_experiment(EXPERIMENT)
settings.update({"mlflow": False})
print("expérience :", EXPERIMENT, "|", len(MODELS), "modèle(s) x", len(RUNS_PLAN), "runs")

## 4. Fine-tunings + évaluations + IC
Chaque run : fine-tuning (sauf k=0), puis évaluation de `last.pt` sur la validation source (seuil du F1),
le test source (oubli) et le test cible figé, puis IC bootstrap (1000 tirages). Un run MLflow par (modèle, k, tirage).

In [ ]:
RUNS = Path("/kaggle/working/runs_fewshot")
EVAL_DATA = {"source_val": ("data_source.yaml", "val"), "source_test": ("data_source.yaml", "test"),
             f"test_{FS['target']}": (f"data_eval_{FS['target']}.yaml", "val")}
epochs = 1 if SMOKE else FS["epochs"]
ci_rows = []

for key in MODELS:
    spec = TRAIN["models"][key]
    cls = YOLO if spec["arch"] == "yolo" else RTDETR
    lr0 = finetune_lr0(spec, FS["lr_scale"], len(CLASSES))
    for r in RUNS_PLAN:
        run_dir = RUNS / key / r.name
        with mlflow.start_run(run_name=f"{key}_{r.name}") as mr:
            mlflow.set_tags({"stage": "S2-fewshot", "smoke": SMOKE, "git_sha": GIT_SHA,
                             "dataset_sha256": DATASET_INFO["manifest_sha256"],
                             "s1_weights_sha256": sha256(S1W[key]), **VERSIONS})
            mlflow.log_params({"model": key, "k": r.k, "subset_seed": r.seed, "train_list": r.list_name,
                               "epochs": 0 if r.k == 0 else epochs, "optimizer": "AdamW", "lr0": lr0,
                               "warmup_bias_lr": FS["warmup_bias_lr"], "train_seed": FS["train_seed"],
                               "batch": TRAIN["batch"], "imgsz": TRAIN["imgsz"]})
            c = COST.set_index("run").loc[r.name]
            mlflow.log_metrics({"cost/n_images": float(c.n_images), "cost/n_boxes": float(c.n_boxes),
                                "cost/hours_low_estimate": float(c.h_low), "cost/hours_high_estimate": float(c.h_high)})
            weights = S1W[key]
            if r.k > 0:
                y = write_fewshot_yaml(DATA, r, CLASSES)
                t0 = time.time()
                m = cls(str(S1W[key]))
                m.train(data=str(y), epochs=epochs, imgsz=TRAIN["imgsz"], batch=TRAIN["batch"], optimizer="AdamW",
                        lr0=lr0, warmup_bias_lr=FS["warmup_bias_lr"], seed=FS["train_seed"],
                        deterministic=TRAIN["deterministic"], patience=0, workers=TRAIN["workers"],
                        amp=TRAIN["amp"], device=0, val=False, plots=False,
                        project=str(RUNS / key), name=r.name, exist_ok=True)
                mlflow.log_metric("time/train_minutes", (time.time() - t0) / 60)
                weights = Path(m.trainer.save_dir) / "weights" / "last.pt"
                torch.cuda.empty_cache()
            for tag, (yml, split) in EVAL_DATA.items():
                cls(str(weights)).val(data=str(DATA / yml), split=split, imgsz=TRAIN["imgsz"], batch=TRAIN["batch"],
                                      conf=CFG["eval"]["conf_map"], save_json=True, plots=False, device=0,
                                      project=str(run_dir), name=f"eval_{tag}", exist_ok=True, verbose=False)
            rows = report(run_dir, DATA, key, n_boot=CFG["eval"]["bootstrap"]["n_resamples"])
            for row in rows:
                row.update(run=r.name, k=r.k, seed=r.seed)
                mlflow.log_metrics({f"ci/{row['eval']}/{m_}": float(row[m_])
                                    for m_ in ("mAP50", "mAP50_lo", "mAP50_hi", "F1", "F1_lo", "F1_hi")})
                for p in run_dir.glob(f"eval_{row['eval']}/predictions.json"):
                    mlflow.log_artifact(str(p), artifact_path=f"predictions/{row['eval']}")
            ci_rows += rows
            tgt = next(x for x in rows if x["eval"] == f"test_{FS['target']}")
            print(f"{key:9s} {r.name:26s} cible mAP50 {tgt['mAP50']:.3f} [{tgt['mAP50_lo']:.3f}, {tgt['mAP50_hi']:.3f}]")

OUT = Path("/kaggle/working/results"); OUT.mkdir(exist_ok=True)
suffix = "_smoke" if SMOKE else ""
pd.DataFrame(ci_rows).to_csv(OUT / f"fewshot_runs{suffix}.csv", index=False)
COST.to_csv(OUT / f"fewshot_cost{suffix}.csv", index=False)

## 5. Courbe « images annotées vs performance » (avec coût estimé) et tableau

In [ ]:
curve = []
for metric in ("mAP50", "F1"):
    for row in aggregate_curve(ci_rows, metric):
        curve.append({"metric": metric, **{k.replace(metric, "value"): v for k, v in row.items()}})
CURVE = pd.DataFrame(curve)
cost_by_k = COST.groupby("k")[["h_low", "h_high"]].mean()
CURVE = CURVE.merge(cost_by_k, left_on="k", right_index=True, how="left")
CURVE.to_csv(OUT / f"fewshot_curve{suffix}.csv", index=False)
display(CURVE[CURVE.metric == "mAP50"].drop(columns="metric"))

png = plot_curve(aggregate_curve(ci_rows, "mAP50"), ci_rows,
                 {int(k): (v.h_low, v.h_high) for k, v in cost_by_k.iterrows()},
                 [(f"test_{FS['target']}", "Pays cible jamais vu (États-Unis)"),
                  ("source_test", "Oubli : test des pays sources")],
                 OUT / f"fewshot_curve{suffix}.png", full_k=pool_size if FS.get("include_full_pool") else None)
with mlflow.start_run(run_name="fewshot_summary" + suffix):
    mlflow.set_tags({"stage": "S2-summary", "git_sha": GIT_SHA, "dataset_sha256": DATASET_INFO["manifest_sha256"]})
    for f in OUT.glob(f"fewshot_*{suffix}.*"):
        mlflow.log_artifact(str(f), artifact_path="results")
from IPython.display import Image
Image(str(png))

## 6. Récupérer les sorties (onglet *Output* de la version)

In [ ]:
!cd /kaggle/working && zip -qr s2_outputs.zip results mlflow.db mlruns 2>/dev/null; ls -lh /kaggle/working

## À me renvoyer
`s2_outputs.zip` (CSV des runs, courbe, coût, MLflow). Aucune conclusion n'est tirée avant ces chiffres et leurs IC.